Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time Series Clustering
# Teil 3: Modeling & Evaluation
***


In [14]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from tslearn.utils import to_time_series_dataset
from sklearn.metrics import adjusted_rand_score
from scipy.stats import kruskal, chi2_contingency
from itertools import combinations

from parameter import INPUT_PATH, OUTPUT_PATH
from funktionen import get_dtw, run_model

warnings.simplefilter(action="ignore", category=FutureWarning)

Daten laden

In [15]:
# Zeitreihe laden
path_temp = INPUT_PATH / "data_timeseries.csv"
df = pd.read_csv(path_temp)

In [16]:
# Zusatzinfos laden
path_temp = INPUT_PATH / "data_add_info.csv"
df_add = pd.read_csv(path_temp)

Leeres Evaluations-DatenFrame und Liste für die Clusterlabels initialisieren

In [17]:
df_evaluate = pd.DataFrame(
    columns=["Durchlauf", "Algorithmus", "Distanzmatrix",
             "Anzahl_Cluster", "Silhouette_Score",
             "Davies_Bouldin_Score", "Dunn_Index_Score"]
)

all_labels = []

***
## Phase 6: Modellierung
***

Im Rahmen der Analyse werden die folgenden Methoden verwendet:

- **k-Means**: Wird als Baseline-Modell verwendet, da der Algorithmus einfach, weit verbreitet und gut interpretierbar ist

- **Agglomeratives Clustering**: Bildet die hierarchische Struktur natürliche Gruppierungen auf verschiedenen Stufen ab

- **k-Medoids**: Clusterzentren sind echte Zeitreihen aus dem Datensatz und keine künstlich berechneten Mittelwerte

- **Fuzzy C-Medoids (FCM)**: Zeitreihen können mehreren Clustern zugeordnet werden , wodurch überlappende Muster realistisch abgebildet werden können

- **Spectral Clustering**: Erkennt nicht-konvexe Clusterformen, welche bei Zeitreihen häufig auftreten können

- **Density-Based Spatial Clustering of Applications with Noise (DBSCAN)**: Clusteranzahl muss vorab nicht festgelegt werden. Zudem werden Ausreißer automatisch identifiziert

Aus dem Grund, dass die Zeitreihen zu groß für die Clusterinmethoden sind, werden fünf Durchläufe mit jeweils 100 Zeitreihen ausgeführt. Bei jedem Durchlauf werden alle Methoden mit der Standard-DTW, DTW mit Sakoe-Chiba-Band und mit Itakura-Parallelogramm ausgeführt.

In [18]:
X = df.values

n_runs = 5  # Anzahl der Durchläufe
n_sample = 100  # Anzahl der Zeitreihen pro Durchlauf

In [19]:
for i in range(n_runs):

    print(f"Durchlauf #{i + 1}:")

    # Zufällig 100 Zeitreihen wählen
    rng = np.random.default_rng(seed=i * 3)
    idx = rng.choice(len(X), n_sample, replace=False)
    X_sample = X[idx]

    # Skalierung
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_sample)

    # tslearn Format
    X_ts = to_time_series_dataset(X_scaled)

    # DTWs bestimmen
    D_dict = get_dtw(X_ts)

    # Modelle anwenden und Ergebnisse messen
    labels_baseline = run_model("baseline", D_dict, df_evaluate, (i + 1), X_scaled, X_ts)
    labels_agglomerative = run_model("agglomerative", D_dict, df_evaluate, (i + 1), X_scaled)
    labels_kmedoids = run_model("kmedoids", D_dict, df_evaluate, (i + 1), X_scaled)
    labels_fcm = run_model("fcm", D_dict, df_evaluate, (i + 1), X_scaled)
    labels_spectral = run_model("spectral", D_dict, df_evaluate, (i + 1), X_scaled)
    labels_dbscan = run_model("dbscan", D_dict, df_evaluate, (i + 1), X_scaled)

    # Labels sammeln
    run_labels = {"card_id": idx, "Durchlauf": i + 1}
    for dist_name, lab in labels_baseline.items():
        run_labels[f"baseline_{dist_name}"] = lab
    for dist_name, lab in labels_agglomerative.items():
        run_labels[f"agglomerative_{dist_name}"] = lab
    for dist_name, lab in labels_kmedoids.items():
        run_labels[f"kmedoids_{dist_name}"] = lab
    for dist_name, lab in labels_fcm.items():
        run_labels[f"fcm_{dist_name}"] = lab
    for dist_name, lab in labels_spectral.items():
        run_labels[f"spectral_{dist_name}"] = lab
    for dist_name, lab in labels_dbscan.items():
        run_labels[f"dbscan_{dist_name}"] = lab
    all_labels.append(pd.DataFrame(run_labels))

    print("-------------------------------------------------------")

df_labels_all = pd.concat(all_labels, ignore_index=True)

Durchlauf #1:
 [Info] Berechne DTW Standard
 [Info] Berechne DTW mit Sakoe-Chiba Band
 [Info] Berechne DTW mit Itakura Parallelogramm
 [Info] Starte Clusteringmethode baseline
 [Info] Starte Clusteringmethode agglomerative


ImportError: cannot import name 'size' from 'scipy._lib._array_api' (/Users/juliaschmid/Desktop/Neuer Ordner 2/DAMI01_DATA01/dami01_env/lib/python3.11/site-packages/scipy/_lib/_array_api.py)

In [8]:
# Ergebnis-Datensatz sortieren
order_algo = ['baseline', 'agglomerative', 'kmedoids', 'fcm', 'spectral', 'dbscan']
order_dtw = ['standard', 'sakoe', 'itakura']

df_evaluate['Algorithmus'] = pd.Categorical(df_evaluate['Algorithmus'], categories=order_algo, ordered=True)
df_evaluate['Distanzmatrix'] = pd.Categorical(df_evaluate['Distanzmatrix'], categories=order_dtw, ordered=True)
df_evaluate = df_evaluate.sort_values(['Algorithmus', 'Distanzmatrix']).reset_index(drop=True)

In [ ]:
df_evaluate

***
## Phase 7: Evaluation
***

### **ARI Stabilität**

In [ ]:
label_cols = [c for c in df_labels_all.columns if c not in ["card_id", "Durchlauf"]]
runs = sorted(df_labels_all["Durchlauf"].unique())

ari_records = []

for col in label_cols:
    for run_a, run_b in combinations(runs, 2):
        df_a = df_labels_all[df_labels_all["Durchlauf"] == run_a][["card_id", col]].dropna()
        df_b = df_labels_all[df_labels_all["Durchlauf"] == run_b][["card_id", col]].dropna()

        # Schnittmenge der card_ids
        merged = df_a.merge(df_b, on="card_id", suffixes=("_a", "_b"))

        if len(merged) < 2:
            continue

        ari = adjusted_rand_score(merged[f"{col}_a"], merged[f"{col}_b"])

        # Algorithmus und Distanzmatrix aus Spaltenname extrahieren
        parts = col.split("_", 1)
        algo = parts[0]
        dist = parts[1] if len(parts) > 1 else "default"

        ari_records.append({
            "Algorithmus": algo,
            "Distanzmatrix": dist,
            "Run_A": run_a,
            "Run_B": run_b,
            "ARI": ari,
            "Overlap": len(merged)
        })

df_ari = pd.DataFrame(ari_records)
df_ari = df_ari.groupby(["Algorithmus", "Distanzmatrix"])["ARI"].mean().reset_index()
df_ari['Algorithmus'] = pd.Categorical(df_ari['Algorithmus'], categories=order_algo, ordered=True)
df_ari['Distanzmatrix'] = pd.Categorical(df_ari['Distanzmatrix'], categories=order_dtw, ordered=True)
df_ari = df_ari.sort_values(['Algorithmus', 'Distanzmatrix']).reset_index(drop=True)
df_ari

**Interpretation**

Die **Baseline (k-Means) Methode** erreicht über alle drei DTW-Distanzen hinweg identische, moderate Stabilitätswerte (ARI = 0.723).

Das **agglomerative Clustering** zeigt über alle drei DTW-Distanzen hinweg eine perfekte Stabilität (ARI = 1.0). Damit liefern diese Verfahren robuste und reproduzierbare Clusterergebnisse.

Das **K-Medoids** Verfahren liefert moderat konsistente Ergebnisse, wobei die Stabilität je nach Distanzmetrik variiert. Die Standard-DTW erzielt den höchsten Wert (ARI = 0.760), gefolgt von der Itakura-DTW (ARI = 0.645), während die Sakoe-DTW die schwächste Stabilität aufweist (ARI = 0.543).

Das **FCM** Verfahren weist über alle DTW-Distanzen hinweg eine durchgehend moderate Stabilität auf. Die Itakura-DTW liefert mit einem ARI von 0.765 die beste Stabilität und liegt damit im moderaten bis guten Bereich. Die Sakoe-DTW erreicht einen ARI von 0.617, während die Standard-DTW mit 0.539 die schwächste Stabilität aufweist.

Das **Spectral Clustering** weist ebenfalls eine hohe und über alle DTW-Distanzmetriken hinweg konstante Stabilität auf (ARI = 0.9). 

Beim **DBSCAN** ist die Stabilität von der gewählten DTW-Distanzmetrik abhängig. Mit der Sakoe-DTW wird die höchste Stabilität erreicht (ARI = 0.9), gefolgt von der Itakura-DTW mit einer guten Konsistenz (ARI = 0.8). Die Standard-DTW führt hingegen zu einer deutlich geringeren Stabilität (ARI = 0.623). 

### **Evaluationskennzahlen**

In [ ]:
print("Besten Ergebnisse nach Metrik")
sil_row = df_evaluate.loc[df_evaluate["Silhouette_Score"].idxmax()]
dunn_row = df_evaluate.loc[df_evaluate["Dunn_Index_Score"].idxmax()]
dav_row = df_evaluate.loc[df_evaluate["Davies_Bouldin_Score"].idxmin()]

print(
    f"Silhouette Score: {sil_row['Algorithmus']} "
    f"{sil_row['Distanzmatrix']} mit "
    f"{sil_row['Silhouette_Score']:.3f}"
    f" im Lauf {sil_row['Durchlauf']}"
)
print(
    f"Dunn Index Score: {dunn_row['Algorithmus']} "
    f"{dunn_row['Distanzmatrix']} mit "
    f"{dunn_row['Dunn_Index_Score']:.3f}"
    f" im Lauf {dunn_row['Durchlauf']}"
)
print(
    f"Davies Bouldin Score: {dav_row['Algorithmus']} "
    f"{dav_row['Distanzmatrix']} mit "
    f"{dav_row['Davies_Bouldin_Score']:.3f}"
    f" im Lauf {dav_row['Durchlauf']}"
)

**Interpretation**

Das **Agglomerative Clustering Modell** liefert mit der standard DTW-Distanz die besten Ergebnisse über alle Metriken.

In [ ]:
metrics = [
    "Silhouette_Score",
    "Davies_Bouldin_Score",
    "Dunn_Index_Score"
]

algorithms = df_evaluate["Algorithmus"].unique()

fig, axes = plt.subplots(len(algorithms), 1, figsize=(10, 5 * len(algorithms)))

for ax, algo in zip(axes, algorithms):
    df_temp = df_evaluate[df_evaluate["Algorithmus"] == algo]

    df_temp = df_temp.melt(
        id_vars=["Durchlauf"],
        value_vars=metrics,
        var_name="Metrik",
        value_name="Score"
    )

    sns.barplot(
        data=df_temp,
        x="Durchlauf",
        y="Score",
        hue="Metrik",
        ax=ax,
        errorbar=None
    )

    ax.set_title(f"Evaluationsmetriken für {algo}")
    ax.set_xlabel("Durchlauf")
    ax.set_ylabel("Score")
    ax.legend(title="Evaluationsmetrike", bbox_to_anchor=(1.01, 1), loc='upper left')


plt.tight_layout()
fig.savefig(OUTPUT_PATH / "Evaluationsmetriken_Durchlaeufe.png")
plt.show()

**Interpretation**

Das **Baseline-Verfahren (k-Means)** liefert über die Durchläufe hinweg schwankende Ergebnisse. Während in den ersten Durchläufen noch ein Silhouette-Score von etwa 0.55 und ein Dunn-Index von etwa 0.60 erreicht werden, sinken diese Werte in den weiteren Durchläufen auf 0.25-0.40. Der Davies-Bouldin-Score schwankt stark zwischen 0.70 und 1.95

Das **agglomeratives Clustering** liefert über alle Durchläufe hinweg konsistent gute Ergebnisse. Die Silhouette-Score (0.55-0.60) und der Dunn-Index (0.55-0.80) sind stabil hoch, während der Davies-Bouldin-Score (0.25-0.35) niedrig bleibt. 

**K-Medoids** zeigt in allen Durchläufe eine eher schwache Clusterqualität. Die Methode erreicht einen Silhouette-Score von lediglich etwa 0.25–0.40, einen Dunn-Index von 0.20–0.35 sowie einen Davies-Bouldin-Score von etwa 1.45–1.75.

**FCM** erreicht die schwächste Leistung mit Davies-Bouldin-Scores von 1.50–1.95, sowie einem niedrigen Silhouette-Score (0.20–0.40) und Dunn-Index (0.20–0.45). Die Methode ist für diese Anwendung eher unzuverlässig.

Das **Spectral Clustering** erreicht einen Silhouette-Score von etwa 0.40–0.50 und einen Dunn-Index von 0.45–0.60, zeigt jedoch deutliche Schwankungen beim Davies-Bouldin-Score (1.00–1.45).

**DBSCAN** liefert eine ähnliche Leistung wie das Spectral Clustering (Silhouette-Score 0.42–0.44, Davies-Bouldin-Score 1.30–1.45, Dunn-Index 0.35–0.50). Die Werte sind relativ konstant über alle Durchläufe hinweg. 

In [ ]:
df_mean = df_evaluate.groupby(["Algorithmus", "Distanzmatrix"]).agg({
    "Silhouette_Score": "mean",
    "Davies_Bouldin_Score": "mean",
    "Dunn_Index_Score": "mean"
}).reset_index()
df_mean

**Interpretation**

Die Durchschnittswerte bestätigen die im vorherigen Abschnitt erkannten Punkte.

In [ ]:
metrics = [
    "Silhouette_Score",
    "Davies_Bouldin_Score",
    "Dunn_Index_Score"
]

fig, axes = plt.subplots(len(metrics), 1, figsize=(10, 5 * len(metrics)))

for ax, metric in zip(axes, metrics):
    sns.barplot(
        data=df_mean,
        x="Algorithmus",
        y=metric,
        hue="Distanzmatrix",
        ax=ax
    )

    ax.set_title(f"{metric} nach Clusteringmethode und Distanzmatrix")
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.legend(title="Distanzmatrix", bbox_to_anchor=(1.01, 1), loc='upper left')


plt.tight_layout()
fig.savefig(OUTPUT_PATH / "Evaluationsmetriken_Clusteringmethode.png", dpi=500, bbox_inches="tight")
plt.show()

**Interpretation**

Der **Silhouette-Score**  ist beim agglomerativen Clustering am höchsten (0.58–0.59), gefolgt von DBSCAN (0.43–0.45) und dem Spectral Clustering (0.43–0.44). Das Baseline-Modell liegt mit Werten von 0.42 knapp darunter, während K-Medoids (0.31–0.33) und das FCM-Verfahren (0.29–0.34) die niedrigsten Silhouette-Scores aufweisen. Bei den meisten Verfahren sind die Unterschiede zwischen den DTW-Varianten gering.

Beim **Davies-Bouldin-Score** erreicht das agglomerative Clustering die niedrigsten Werte (0.30–0.32), was auf eine gute Clustertrennung hindeutet. Das Spectral Clustering (1.25–1.31) und DBSCAN (1.25–1.45) liegen im mittleren Bereich. Das Baseline-Modell weist Werte zwischen 1.38 und 1.55 auf. K-Medoids (1.53–1.70) und das FCM-Verfahren (1.60–1.73) erzielen die höchsten Davies-Bouldin-Scores.

Der **Dunn-Index** ist beim agglomerativen Clustering am höchsten (0.68–0.71), gefolgt vom Spectral Clustering (0.49–0.55) und DBSCAN (0.41–0.53). Das Baseline-Modell erreicht Werte von 0.37–0.40. K-Medoids (0.29–0.31) und das FCM-Verfahren (0.29–0.39) weisen die niedrigsten Dunn-Index-Werte auf. Beim Dunn-Index liefert die Standard-DTW für die meisten Verfahren die besten Ergebnisse, mit Ausnahme des FCM-Verfahrens, bei dem die Sakoe-DTW den höchsten Wert erzielt.

### **Kontextanalyse**

In [15]:
# Labels des ersten Durchlaufs genauer betrachten
df_labels = df_labels_all[df_labels_all["Durchlauf"] == 1].drop(columns=["Durchlauf"])

df_sample_meta = df.iloc[df_labels.index]  # Zeitreihen auf Sample selektieren
df_cluster_profile = df.iloc[df_labels["card_id"].values]  # card_id = Zeilenindex in df
df_cluster_profile = df_labels.merge(
    df_cluster_profile
    .reset_index(drop=True)
    .assign(card_id=df_labels["card_id"].values),
    on="card_id"
)
df_cluster_profile = df_cluster_profile.merge(df_add, how="left", on="card_id")

# # Cluster-Methoden
cluster_methods = [c for c in df_labels.columns if c != "card_id"]

# Methoden gruppieren
groups = {
    "standard": [m for m in cluster_methods if m.endswith("standard")],
    "sakoe": [m for m in cluster_methods if "sakoe" in m],
    "itakura": [m for m in cluster_methods if "itakura" in m],
}

df_combined = df_labels.merge(df_add, on='card_id')

### Clusterverteilung pro Methode

In [ ]:
# Cluster-Verteilung
for col in [c for c in cluster_methods]:
    counts = df_labels[col].value_counts().sort_index().reset_index()
    counts.columns = ["Cluster", "Anzahl"]
    counts["Anteil (%)"] = (counts["Anzahl"] / len(df_labels) * 100).round(1)
    print(col)
    print(counts.to_string(index=False))

**Interpretation**

Das **Baseline-Verfahren (k-Means)** liefert über alle drei DTW-Distanzen hinweg ein identisches Ergebnis. So werden 98 Prozent der Zeitreihen einem Cluster zugeordnet, während lediglich zwei Prozent auf einen zweiten Cluster entfallen.

Das **Agglomerative Clustering**  ordnet über alle drei DTW-Distanzen hinweg 99 Prozent der Zeitreihen einem einzigen Cluster zu, während lediglich eine einzelne Zeitreihe einem zweiten Cluster zugewiesen wird.

Eine sinnvolle Aufteilung liefert das **K-Medoids Verfahren**. Mit der Standard-DTW ergibt sich eine Verteilung von 49 zu 51 Prozent, mit der Sakoe-DTW von 42 zu 58 Prozent und mit der Itakura-DTW von 37 zu 63 Prozent.

Beim **FCM-Verfahren** zeigt sich eine ähnlich ausgewogene Aufteilung. Mit der Standard-DTW ergibt sich eine Verteilung von 53 zu 47 Prozent, mit der Sakoe-DTW von 58 zu 42 Prozent und mit der Itakura-DTW von 55 zu 45 Prozent.

Das **Spectral Clustering** verteilt die Daten über alle Distanzmetriken hinweg stark ungleichmäßig. Mit der Standard-DTW werden 94 Prozent der Zeitreihen einem Cluster zugeordnet, während die verbleibenden sechs Prozent auf zwei kleinere Cluster entfallen. Bei der Sakoe-DTW entsteht eine Verteilung von 97 zu 3 Prozent und bei der Itakura-DTW werden 95 Prozent einem Cluster zugewiesen, während zwei weitere Cluster lediglich 4 und 1 Prozent umfassen.

Auch das **DBSCAN-Verfahren** fasst den Großteil der Daten in einen einzigen Cluster zusammen und kennzeichnet lediglich 7 bis 13 Prozent der Zeitreihen als Rauschen (Cluster -1).

### Zeitreihen-Cluster pro Methode

In [ ]:
# Zeitreihe-Cluster plotten
time_cols = []
for col in df_cluster_profile.columns:
    try:
        pd.to_datetime(col)
        time_cols.append(col)
    except Exception:
        continue

# Zeitindex einmal erstellen
time_index = pd.to_datetime(time_cols)

for group_name, methods in groups.items():
    if not methods:
        continue

    n_methods = len(methods)
    max_clusters = max(len(df_cluster_profile[m].unique()) for m in methods)

    fig = plt.figure(figsize=(5 * max_clusters, 4 * n_methods))
    gs = gridspec.GridSpec(n_methods, max_clusters)

    for row_idx, method in enumerate(methods):
        clusters = sorted(df_cluster_profile[method].unique())
        palette = sns.color_palette("tab10", len(clusters))

        for col_idx, (cluster, color) in enumerate(zip(clusters, palette)):
            ax = fig.add_subplot(gs[row_idx, col_idx])

            df_cluster = df_cluster_profile[df_cluster_profile[method] == cluster]

            # Einzelne Zeitreihen
            for idx, row in df_cluster.iterrows():
                ax.plot(time_index, row[time_cols], color=color, alpha=0.3)

            # Mittelwert + Rolling Mean
            mean_vals = df_cluster[time_cols].mean().values
            mean_series = pd.Series(mean_vals).rolling(window=30, center=True).mean()

            ax.plot(time_index, mean_series, color="black", linewidth=2)
            ax.set_title(f"Zeitreihencluster - {method} - {cluster}")
            ax.tick_params(axis="x", rotation=45)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(OUTPUT_PATH / f"Zeitreihencluster_{group_name}.png")
    plt.show()

**Interpetation**

Das **Baseline-Modell** liefert konstant zwei Cluster über alle drei DTW-Varianten hinweg. Cluster 0 enthält dabei die große Mehrheit der Zeitreihen im niedrigen Niveau, während Cluster 1 die Zeitreihen mit höherem Niveau und einzelnen starken Ausreißern umfasst.

Auch das **agglomerative Clustering** liefert ebenfalls zwei Cluster bei allen drei DTW-Varianten.  Cluster 0 ist deutlich größer und enthält die Mehrheit der Zeitreihen mit moderater Streuung. Cluster 1 beinhaltet dagegen lediglich eine einzelne Zeitreihe mit deutlich höherem Niveau und vereinzelten extremen Spitzen.

Beim **k-Medoids** entstehen bei allen drei DTW-Varianten jeweils zwei Cluster mit einer vergleichsweise ausgewogenen Aufteilung. Cluster 0 enthält dabei die volatileren Zeitreihen mit höheren Ausschlägen, während Cluster 1 die gleichmäßigeren Zeitreihen mit niedrigerem Niveau umfasst.

Das **FCM** erzeugt bei allen Varianten zwei Cluster, welche nach Volatilität getrennt sind. Cluster 0 enthält dabei die niedrigeren, gleichmäßigeren Zeitreihen, während Cluster 1 die volatileren Zeitreihen mit größeren Ausschlägen umfasst.

Beim **spectral Clustering** zeigt sich die größte Varianz zwischen den DTW-Varianten. Die Standard-DTW und die Itakura-DTW erzeugen jeweils drei Cluster, während die Sakoe-DTW nur zwei Cluster liefert. Die zusätzlichen Cluster enthalten oft nur wenige Zeitreihen mit speziellen Mustern, etwa einzelne Ausreißer oder Zeitreihen mit Strukturbrüchen. Spectral Clustering zeigt deutliche Unterschiede abhängig von der Wahl der DTW-Variante.

**DBSCAN** erzeugt bei allen drei Varianten das Cluster -1 und Cluster 0. Cluster -1 enthält die Zeitreihen mit extremen Ausschlägen, die DBSCAN als Rauschen behandelt. Cluster 0 umfasst die übrigen, gleichmäßigeren Zeitreihen. Die Ergebnisse sind über alle DTW-Varianten hinweg konsistent.

### Cluster-Verteilung der numerischen Zusatzvariablen

In [ ]:
# Numerische Variablen auswählen
numeric_cols = df_combined.select_dtypes(include=np.number).columns.tolist()

# Nicht relevante numerische Spalten entfernen
numeric_cols = [
    col for col in numeric_cols
    if col not in list(cluster_methods) + ["card_id", "client_id"]
]

# Für jede numerische Variable Boxplot erstellen
for col in numeric_cols:

    # Daten ins Long-Format bringen:
    # Eine Zeile pro (Beobachtung, Methode)
    df_long = df_combined[cluster_methods + [col]].melt(
        id_vars=[col],
        value_vars=cluster_methods,
        var_name="method",
        value_name="cluster"
    )

    # Cluster als String formatieren (wichtig für Farbzuordnung)
    df_long["cluster"] = df_long["cluster"].astype(str)

    # Alle Cluster sortieren (numerisch, falls möglich)
    all_clusters = sorted(
        df_long["cluster"].unique(),
        key=lambda x: int(x) if x.lstrip('-').isdigit() else x
    )

    # Farbpalette definieren
    palette = sns.color_palette("tab10", len(all_clusters))

    # Plot erstellen
    fig, ax = plt.subplots(
        figsize=(max(14, 2 * len(cluster_methods)), 6)
    )

    sns.boxplot(
        data=df_long,
        x="method",
        y=col,
        hue="cluster",
        palette=palette,
        ax=ax,
        order=cluster_methods,
        hue_order=all_clusters
    )

    # Titel und Achsenbeschriftung
    ax.set_title(f"Boxplots - {col}")
    ax.set_xlabel("")
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    fig.savefig(OUTPUT_PATH / f"Boxplot_{col}.png")
    plt.show()

In [ ]:
BOLD_START = "\033[1m"
BOLD_END = "\033[0m"

for method in cluster_methods:
    print(f"{BOLD_START}{method}{BOLD_END}")
    for var in numeric_cols:
        print(var.upper())
        stats = df_combined.groupby(method)[var].agg([
            'count', 'min', 'max', 'mean', 'median',
        ]).round(2)
        print(stats)
        print("")
    
    print("-------------------------------------------------------")

**Interpretation**

Das **Baseline-Modell**
Das Baseline-Modell isoliert lediglich zwei Zeitreihen (2 %) in einem separaten Cluster. Da für diesen kleinen Cluster keine aussagekräftigen Statistiken vorliegen, lässt sich keine inhaltliche Differenzierung anhand der numerischen Variablen ableiten. Die verschiedenen DTW-Distanzen liefern identische Ergebnisse.

Beim **agglomerativen Clustering-Verfahren** fallen alle Zeitreihen über alle DTW-Varianten in einen Cluster. Dadurch entsteht keine Differenzierung bei den numerischen Variablen.

Im Gegensatz dazu liefert das **k-Medoid Verfahren** ein deutlich differenzierteres Ergebnis. Beim K-Medoids mit der Standard-DTW entstehen zwei Cluster, wobei Cluster 0 einen niedrigeren total_debt aufweist (Median 40.161), weniger num_credit_cards (Median 3) und einen höheren birth_month (Median 10). Die Sakoe-DTW und Itakura-DTW liefern jeweils ebenfalls zwei Cluster mit einem ähnlichen Muster.

Das **FCM** hat bei allen DTW-Distanzen zwei Cluster. Dabei hat Cluster 0 das höhere credit_limit (Median 12.896) und den höheren total_debt (Median 60.155), während Cluster 1 einen besseren credit_score aufweist (Median 712) und einen höheren birth_month (Median 10). Die Sakoe-DTW und Itakura-DTW zeigen dieselbe Struktur.

**Spectral Clustering** mit der Standard-DTW und Itakura-DTW erzeugt jeweils drei Cluster mit ungleicher Verteilung. Dabei enthält der kleinste Cluster nur eine einzelne Zeitreihe, die sich durch einen höheren credit_score (735), ein höheres credit_limit (18.173), keinen total_debt (0) sowie einen hohen birth_month (11) auszeichnet. Die Sakoe-DTW bildet hingegen nur zwei Cluster, wobei der kleinere Cluster dieselbe einzelne Zeitreihe mit denselben Merkmalen enthält.

Durch das **DBSCAN-Verfahren** werden neben dem Hauptcluster nur Ausreißer identifiziert (Cluster -1). Bei der Standard-DTW handelt es sich um eine einzelne Zeitreihe. Bei der Sakoe- und Itakura-DTW umfasst Cluster -1 jeweils zwei Zeitreihen, die sich durch einen jüngeren Jahrgang (Mean birth_year 1971), einen höheren birth_month (Median 10,5), einen leicht höheren credit_score (Median 723,5) sowie einen niedrigeren total_debt (Mean 39.416) auszeichnen.


### Cluster-Verteilung der kategorischen Zusatzvariablen

In [ ]:
# Kategorische-Variablen (Gestackter Bar Plot)
cat_cols = df_combined.select_dtypes(exclude=np.number).columns.tolist()
cat_cols = [col for col in cat_cols if col not in list(cluster_methods) + ["card_id", "client_id"]]

cluster_methods_grouped = {
    "baseline": ["baseline_standard", "baseline_sakoe", "baseline_itakura"],
    "agglomerative": ["agglomerative_standard", "agglomerative_sakoe", "agglomerative_itakura"],
    "kmedoids": ["kmedoids_standard", "kmedoids_sakoe", "kmedoids_itakura"],
    "fcm": ["fcm_standard", "fcm_sakoe", "fcm_itakura"],
    "spectral": ["spectral_standard", "spectral_sakoe", "spectral_itakura"],
    "dbscan": ["dbscan_standard", "dbscan_sakoe", "dbscan_itakura"],
}

for col in cat_cols:
    unique_vals = sorted(df_combined[col].dropna().unique())
    n_dtw = len(cluster_methods_grouped)
    n_method = max(len(v) for v in cluster_methods_grouped.values())
    
    fig, axes = plt.subplots(n_dtw, n_method, figsize=(7 * n_method, 5 * n_dtw), sharey=True)
    
    for row, (base, variants) in enumerate(cluster_methods_grouped.items()):
        for col_idx in range(n_method):
            ax = axes[row][col_idx]
            method = variants[col_idx]
            ct = pd.crosstab(df_combined[col], df_combined[method])
            ct = ct.reindex(unique_vals, fill_value=0)
            ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab10")
            dtw_variant = method.rsplit("_", 1)[-1]
            ax.set_title(f"Bar Chart - {base} – {dtw_variant}")
            ax.tick_params(axis="x", rotation=45)
            ax.set_xlabel(col if row == n_dtw - 1 else "")
            ax.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc='upper left')

    
    plt.tight_layout()
    fig.savefig(OUTPUT_PATH / f"BarChart_{col}.png")
    plt.show()

**Interpetation**

Das **Baseline-Model** erzeugt über alle drei DTW-Varianten hinweg konsistent zwei Cluster. Da Cluster 0 mit 98 % der Zeitreihen nahezu alle Daten enthält, entspricht die Variablenverteilung der Gesamtverteilung. Eine merkmalspezifische Differenzierung ist nicht erkennbar.

Beim **agglomerative Clustering** gibt es nur ein einziges Cluster, unabhängig von Merkmal und DTW-Variante.

Eine deutliche Trennung ist bei dem **k-Medoids** Verfahren zu erkennen. Bei der Standard-DTW enthält Cluster 0 anteilig mehr Food_and_Beverage-Transaktionen, während Cluster 1 stärker von Retail_General und Professional_Services geprägt ist. Beim Geschlecht zeigt sich eine leichte Tendenz. Das Cluster 0 enthält anteilig mehr Männer und Cluster 1 mehr Frauen. Die Sakoe- und Itakura-DTW erzeugen ein ähnliches Muster mit vergleichbarer Aufteilung.

Das **FCM** zeigt eine leichte Tendenzen in der Clusterzusammensetzung. Bei der Standard-DTW enthält Cluster 1 anteilig mehr Food_and_Beverage-Einträge und einen höheren birth_month, während Cluster 0 stärker von Retail_General geprägt ist. Beim Geschlecht ist Cluster 0 etwas stärker weiblich besetzt. Die Unterschiede sind insgesamt jedoch gering.

Beim **spectral Clustering** mit der Standard-DTW und Sakoe-DTW domniert ein Cluster mit über 45 Zeitreihen. Die Variablenverteilung dieses Clusters entspricht weitgehend der Gesamtverteilung. Die zusätzlichen kleineren Cluster enthalten jeweils nur einzelne Zeitreihen, sodass eine aussagekräftige Charakterisierung nicht möglich ist.

Mit dem **DBSCAN-Verfahren** werden neben dem Hauptcluster Ausreißer identifiziert (Cluster -1). Die Variablenverteilung ist nahezu identisch mit der Gesamtverteilung, weswegen eine Charakterisierug nicht möglich ist. 


### Signifikantstest

In [21]:
results = []
alpha = 0.05

# Kategoriale Merkmale
for m  in cluster_methods:
    for c in cat_cols:
        table_crosstab = pd.crosstab(df_combined[m], df_combined[c])
        chi2, p, dof, expected = chi2_contingency(table_crosstab)  # Chi-Quadrat-Kontingenztest
        results.append({
            "Methode": m,
            "Merkmal": c,
            "Test": "Chi-Quadrat",
            "Statistik": round(chi2, 2),
            "p-Wert": round(p, 4),
            "Signifikant": "Ja" if p < alpha else "Nein"
        })

# Numerische Merkmale
for m in cluster_methods:
    for c in numeric_cols:
        groups = [g[c].dropna().values for _, g in df_combined.groupby(m)]
        if len(groups) >= 2:
            stat, p = kruskal(*groups)
            results.append({
                "Methode": m,
                "Merkmal": c,
                "Test": "Kruskal",
                "Statistik": round(stat, 2),
                "p-Wert": round(p, 4),
                "Signifikant": "Ja" if p < alpha else "Nein"
            })

df_results = pd.DataFrame(results)

In [ ]:
df_results[df_results["Signifikant"] == "Ja"]

**Interpretation**

Signifikante Clusterunterschiede zeigen sich bei **k-Medoids** (Sakoe: card_brand, birth_month, Itakura: num_cards_issued, birth_month), **DBSCAN** (Sakoe und Itakura: description_category) sowie **FCM** (Sakoe: birth_month, card_brand, Itakura: birth_month). Die übrigen Methoden weisen keine signifikanten Unterschiede auf.

***
***